# Análisis de Emisiones GHG y Huella de Carbono
## Colombia en el Contexto Latinoamericano — Metodología GHG Protocol

**Dataset:** Our World in Data — CO2 and Greenhouse Gas Emissions  
**Fuente:** Hannah Ritchie, Pablo Rosado & Max Roser (2023). *CO₂ and Greenhouse Gas Emissions*. Our World in Data. https://ourworldindata.org/co2-and-greenhouse-gas-emissions  
**Autora:** Wendy J. Hernández  
**Perfil:** Ingeniería Química · Especialización Ambiental · Data Analytics  
**GitHub:** github.com/wjhernandez

---

## Contexto

El cambio climático es el desafío ambiental más urgente del siglo XXI. El seguimiento y reporte de emisiones de GHG (Greenhouse Gas — Gases de Efecto Invernadero) es el primer paso para cualquier estrategia de descarbonización empresarial o nacional. El estándar internacional más utilizado para este propósito es el **GHG Protocol (Greenhouse Gas Protocol — Protocolo de Gases de Efecto Invernadero)**, desarrollado por el World Resources Institute (WRI) y el World Business Council for Sustainable Development (WBCSD).

Colombia se comprometió bajo el Acuerdo de París a reducir sus emisiones GHG en un **51% para 2030** respecto al escenario tendencial, uno de los compromisos NDC (Nationally Determined Contributions — Contribuciones Determinadas a Nivel Nacional) más ambiciosos de América Latina.

### Preguntas de análisis

> 1. ¿Cuál es la tendencia histórica de emisiones de CO2 en Colombia y cómo se compara con la región latinoamericana?
> 2. ¿Qué factores explican la intensidad de carbono de Colombia (emisiones por unidad de PIB)?
> 3. ¿Está Colombia en trayectoria para cumplir su NDC al 2030?
> 4. ¿Cómo se distribuyen las emisiones per cápita entre los países de la región?

---
## 0. Configuración del Entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de colores con identidad ambiental
PALETTE = {
    'colombia':     '#2d7d9a',
    'latam':        '#f4a261',
    'mundo':        '#e63946',
    'verde':        '#52b788',
    'alerta':       '#e63946',
    'neutro':       '#adb5bd'
}

# Países de América Latina para comparación regional
LATAM = [
    'Colombia', 'Brazil', 'Mexico', 'Argentina', 'Chile',
    'Peru', 'Ecuador', 'Venezuela', 'Bolivia', 'Paraguay',
    'Uruguay', 'Costa Rica', 'Panama', 'Guatemala', 'Honduras'
]

print('Entorno configurado correctamente.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

---
## 1. Carga y Exploración del Dataset

El dataset de Our World in Data (OWID) es la fuente de referencia global para datos de emisiones GHG. Integra datos del Global Carbon Project, Climate Watch, BP Statistical Review y otras fuentes primarias. Se actualiza anualmente y es ampliamente citado por el IPCC (Intergovernmental Panel on Climate Change — Panel Intergubernamental sobre Cambio Climático) y organismos internacionales.

In [ ]:
import os

# ============================================================
# CARGA DEL DATASET — OWID CO2 and Greenhouse Gas Emissions
# Actualizado con Global Carbon Budget 2024 (datos hasta 2023)
# Fuente: Ritchie, H., Rosado, P. & Roser, M. (2023). Our World in Data.
# ============================================================

# INSTRUCCIONES DE DESCARGA:
# 1. Ve a este link en tu navegador y descarga el archivo:
#    https://github.com/owid/co2-data/blob/master/owid-co2-data.csv
#    Haz clic en el botón "Download raw file" (ícono de descarga)
# 2. Guarda el archivo en la misma carpeta donde está este notebook
# 3. Ejecuta esta celda

ARCHIVO_LOCAL = 'owid-co2-data.csv'

# Verificación de existencia del archivo
if not os.path.exists(ARCHIVO_LOCAL):
    print(f'ERROR: No se encuentra el archivo {ARCHIVO_LOCAL}')
    print()
    print('Descárgalo desde:')
    print('  https://github.com/owid/co2-data/blob/master/owid-co2-data.csv')
    print('  Clic en "Download raw file" y guárdalo en esta carpeta.')
else:
    df_raw = pd.read_csv(ARCHIVO_LOCAL)
    print('=== DIMENSIONES DEL DATASET ===')
    print(f'Registros: {df_raw.shape[0]:,}   |   Variables: {df_raw.shape[1]}')
    print(f'Período: {df_raw["year"].min()} — {df_raw["year"].max()}')
    print(f'Países/Regiones: {df_raw["country"].nunique()}')
    print()
    print('=== PRIMERAS 5 FILAS ===')
    print(df_raw.head())

In [ ]:
# Diccionario de variables clave
variables_clave = {
    'country':                  'País o región',
    'year':                     'Año',
    'co2':                      'Emisiones CO2 totales (MtCO2 — megatoneladas de CO2)',
    'co2_per_capita':           'Emisiones CO2 per cápita (tCO2/habitante)',
    'co2_per_gdp':              'Intensidad de carbono (kgCO2 por USD de PIB)',
    'coal_co2':                 'Emisiones CO2 por carbón (MtCO2)',
    'oil_co2':                  'Emisiones CO2 por petróleo (MtCO2)',
    'gas_co2':                  'Emisiones CO2 por gas natural (MtCO2)',
    'cement_co2':               'Emisiones CO2 por cemento (MtCO2)',
    'methane':                  'Emisiones CH4 — metano (MtCO2eq)',
    'nitrous_oxide':            'Emisiones N2O — óxido nitroso (MtCO2eq)',
    'total_ghg':                'Emisiones totales GHG (MtCO2eq)',
    'energy_per_capita':        'Consumo energético per cápita (kWh/habitante)',
    'share_global_co2':         'Participación en emisiones globales (%)',
    'cumulative_co2':           'Emisiones CO2 acumuladas históricas (MtCO2)',
    'population':               'Población',
    'gdp':                      'PIB en USD constantes 2011 (PPP)'
}

print('=== VARIABLES CLAVE DEL DATASET ===')
for var, desc in variables_clave.items():
    disponible = 'OK' if var in df_raw.columns else 'NO DISPONIBLE'
    print(f'  [{disponible}]  {var:<30} {desc}')

In [ ]:
# Filtro: período moderno (1990-2022) y países de interés
AÑO_INICIO = 1990
AÑO_FIN = 2022

df = df_raw[
    (df_raw['year'] >= AÑO_INICIO) &
    (df_raw['year'] <= AÑO_FIN) &
    (df_raw['country'].isin(LATAM + ['World']))
].copy()

# Dataset específico de Colombia
df_col = df[df['country'] == 'Colombia'].copy()

# Dataset de América Latina sin Colombia
df_latam = df[df['country'].isin(LATAM)].copy()

print(f'Registros filtrados: {df.shape[0]:,}')
print(f'Período: {AÑO_INICIO} — {AÑO_FIN}')
print(f'Colombia — registros disponibles: {df_col.shape[0]}')
print()
print('Variables con datos para Colombia:')
nulos_col = df_col[list(variables_clave.keys())].isnull().sum()
print(nulos_col[nulos_col < len(df_col)].to_string())

---
## 2. Análisis Exploratorio — Tendencia Histórica de Emisiones

El análisis de tendencias históricas es el punto de partida de cualquier inventario GHG (Greenhouse Gas — Gases de Efecto Invernadero). Permite identificar patrones, quiebres de tendencia y la velocidad de cambio en las emisiones — información esencial para establecer líneas base y evaluar el impacto de políticas ambientales.

In [ ]:
# --- 2.1 Tendencia de emisiones CO2 totales — Colombia ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Emisiones totales CO2 Colombia
axes[0].fill_between(df_col['year'], df_col['co2'],
                     alpha=0.3, color=PALETTE['colombia'])
axes[0].plot(df_col['year'], df_col['co2'],
             color=PALETTE['colombia'], linewidth=2.5, label='Colombia')
axes[0].set_title('Emisiones CO2 Totales — Colombia\n(1990–2022)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('MtCO2 (megatoneladas de CO2)')
axes[0].legend()

# Comparación regional: top 5 países LATAM por emisiones 2022
top5_latam = df_latam[df_latam['year'] == 2022].nlargest(5, 'co2')['country'].tolist()
colors_top5 = [PALETTE['colombia'] if c == 'Colombia' else PALETTE['neutro'] for c in top5_latam]

for i, pais in enumerate(top5_latam):
    subset = df_latam[df_latam['country'] == pais]
    color = PALETTE['colombia'] if pais == 'Colombia' else f'C{i}'
    lw = 2.5 if pais == 'Colombia' else 1.2
    axes[1].plot(subset['year'], subset['co2'],
                 label=pais, linewidth=lw, color=color)

axes[1].set_title('Emisiones CO2 — Top 5 América Latina\n(1990–2022)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('MtCO2')
axes[1].legend(fontsize=9)

plt.suptitle('Tendencia Histórica de Emisiones de CO2',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_tendencia_co2.png', dpi=150, bbox_inches='tight')
plt.show()

# Estadísticas clave
co2_1990 = df_col[df_col['year'] == 1990]['co2'].values[0]
co2_2022 = df_col[df_col['year'] == 2022]['co2'].values[0]
crecimiento = (co2_2022 - co2_1990) / co2_1990 * 100
print(f'Colombia — Emisiones CO2:')
print(f'  1990: {co2_1990:.1f} MtCO2')
print(f'  2022: {co2_2022:.1f} MtCO2')
print(f'  Crecimiento 1990-2022: {crecimiento:.1f}%')

In [ ]:
# --- 2.2 Composición de emisiones por fuente — Colombia ---
fuentes = ['coal_co2', 'oil_co2', 'gas_co2', 'cement_co2']
fuentes_labels = [
    'Carbón (Coal)',
    'Petróleo (Oil)',
    'Gas Natural (Gas)',
    'Cemento (Cement)'
]
colores_fuentes = ['#1a3a4a', '#2d7d9a', '#52b788', '#f4a261']

df_col_fuentes = df_col[['year'] + fuentes].dropna().copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de área apilada — composición histórica
axes[0].stackplot(
    df_col_fuentes['year'],
    [df_col_fuentes[f] for f in fuentes],
    labels=fuentes_labels,
    colors=colores_fuentes,
    alpha=0.85
)
axes[0].set_title('Composición de Emisiones CO2 por Fuente — Colombia',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('MtCO2')
axes[0].legend(loc='upper left', fontsize=9)

# Gráfico de dona — composición 2022
datos_2022 = df_col[df_col['year'] == 2022][fuentes].values[0]
datos_2022_clean = [max(0, v) for v in datos_2022]
wedges, texts, autotexts = axes[1].pie(
    datos_2022_clean,
    labels=fuentes_labels,
    colors=colores_fuentes,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.6)
)
axes[1].set_title('Composición de Emisiones CO2 — Colombia 2022',
                  fontsize=11, fontweight='bold')

plt.suptitle('Análisis de Fuentes de Emisión — Colombia',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2_composicion_fuentes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Intensidad de Carbono y Emisiones Per Cápita

La **intensidad de carbono** (CO2 por unidad de PIB — Producto Interno Bruto) es el indicador que usa el GHG Protocol para medir la eficiencia de una economía en términos de emisiones. Una economía puede crecer en emisiones absolutas pero mejorar su intensidad de carbono si su economía crece más rápido que sus emisiones.

Las **emisiones per cápita** son el indicador de equidad climática — permiten comparar la responsabilidad individual de cada ciudadano en el total de emisiones globales, independientemente del tamaño del país.

In [ ]:
# --- 3.1 Intensidad de carbono — Colombia vs LATAM ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Intensidad de carbono Colombia
axes[0].plot(df_col['year'], df_col['co2_per_gdp'],
             color=PALETTE['colombia'], linewidth=2.5, label='Colombia')
axes[0].fill_between(df_col['year'], df_col['co2_per_gdp'],
                     alpha=0.2, color=PALETTE['colombia'])

# Promedio LATAM
latam_avg = df_latam.groupby('year')['co2_per_gdp'].mean().reset_index()
axes[0].plot(latam_avg['year'], latam_avg['co2_per_gdp'],
             color=PALETTE['latam'], linewidth=1.8,
             linestyle='--', label='Promedio LATAM')

axes[0].set_title('Intensidad de Carbono\n(kgCO2 por USD de PIB)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('kgCO2 / USD PIB')
axes[0].legend()

# Emisiones per cápita — comparación regional 2022
latam_2022 = df_latam[df_latam['year'] == 2022][['country', 'co2_per_capita']].dropna()
latam_2022 = latam_2022.sort_values('co2_per_capita', ascending=True)
colors_bar = [PALETTE['colombia'] if c == 'Colombia'
              else PALETTE['verde'] for c in latam_2022['country']]

bars = axes[1].barh(latam_2022['country'], latam_2022['co2_per_capita'],
                    color=colors_bar, edgecolor='white', height=0.7)
for bar, val in zip(bars, latam_2022['co2_per_capita']):
    axes[1].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

axes[1].set_title('Emisiones CO2 Per Cápita — América Latina 2022\n(tCO2 por habitante)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('tCO2 per cápita')

plt.suptitle('Intensidad de Carbono y Emisiones Per Cápita',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_intensidad_percapita.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. GHG Protocol — Clasificación por Alcance (Scope)

El GHG Protocol (Greenhouse Gas Protocol — Protocolo de Gases de Efecto Invernadero) clasifica las emisiones en tres alcances (Scopes) para evitar el doble conteo y atribuir correctamente la responsabilidad de las emisiones:

- **Scope 1:** Emisiones directas — combustión de combustibles propios, procesos industriales, vehículos propios
- **Scope 2:** Emisiones indirectas por energía — consumo de electricidad y calor comprado a terceros
- **Scope 3:** Otras emisiones indirectas — cadena de suministro, uso del producto, viajes de negocio

En este análisis se trabaja con los equivalentes nacionales: emisiones por combustión de combustibles fósiles (Scope 1 nacional) y se estima el Scope 2 usando el factor de emisión de la red eléctrica colombiana.

In [ ]:
# --- 4.1 Clasificación GHG Protocol para Colombia ---

df_ghg = df_col[['year', 'co2', 'coal_co2', 'oil_co2', 'gas_co2',
                  'cement_co2', 'methane', 'nitrous_oxide', 'total_ghg']].copy()

# Scope 1: emisiones directas por combustión y procesos industriales
df_ghg['scope1_co2'] = df_ghg[['coal_co2', 'oil_co2', 'gas_co2', 'cement_co2']].sum(axis=1)

# Scope 2 estimado: consumo eléctrico × factor de emisión red colombiana
# Factor de emisión red eléctrica Colombia: 0.214 tCO2/MWh (UPME, 2022)
FACTOR_EMISION_COL = 0.214  # tCO2/MWh

# Estimación consumo eléctrico desde energy_per_capita y población
df_ghg_energia = df_col[['year', 'energy_per_capita', 'population']].copy()
df_ghg = df_ghg.merge(df_ghg_energia, on='year', how='left')

# Consumo eléctrico total estimado (MWh) — se asume 30% del consumo energético es eléctrico
df_ghg['consumo_electrico_mwh'] = (
    df_ghg['energy_per_capita'] * df_ghg['population'] * 0.30 / 1000
)
df_ghg['scope2_estimado_mtco2'] = (
    df_ghg['consumo_electrico_mwh'] * FACTOR_EMISION_COL / 1e6
)

# GHG total = CO2 + CH4 + N2O (en CO2eq)
df_ghg['ghg_no_co2'] = df_ghg['total_ghg'] - df_ghg['co2']

print('=== INVENTARIO GHG COLOMBIA — ÚLTIMOS 5 AÑOS ===')
cols_mostrar = ['year', 'co2', 'scope1_co2', 'scope2_estimado_mtco2',
                'methane', 'nitrous_oxide', 'total_ghg']
print(df_ghg[cols_mostrar].tail(5).round(2).to_string(index=False))
print()
print(f'Factor de emisión red eléctrica Colombia: {FACTOR_EMISION_COL} tCO2/MWh (UPME, 2022)')

In [ ]:
# --- 4.2 Visualización del inventario GHG por gas ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Emisiones totales GHG por tipo de gas
df_ghg_plot = df_ghg.dropna(subset=['methane', 'nitrous_oxide', 'co2'])

axes[0].stackplot(
    df_ghg_plot['year'],
    df_ghg_plot['co2'],
    df_ghg_plot['methane'],
    df_ghg_plot['nitrous_oxide'],
    labels=[
        'CO2 (Dióxido de Carbono)',
        'CH4 (Metano)',
        'N2O (Óxido Nitroso)'
    ],
    colors=['#1a3a4a', '#52b788', '#f4a261'],
    alpha=0.85
)
axes[0].set_title('Inventario GHG Colombia por Gas\n(MtCO2eq — metodología GHG Protocol)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('MtCO2eq (megatoneladas de CO2 equivalente)')
axes[0].legend(fontsize=9)

# Scope 1 vs Scope 2 estimado
df_scope = df_ghg.dropna(subset=['scope1_co2', 'scope2_estimado_mtco2'])
axes[1].stackplot(
    df_scope['year'],
    df_scope['scope1_co2'],
    df_scope['scope2_estimado_mtco2'],
    labels=['Scope 1 (Emisiones Directas)', 'Scope 2 (Energía Indirecta — Estimado)'],
    colors=[PALETTE['colombia'], PALETTE['verde']],
    alpha=0.85
)
axes[1].set_title('Clasificación GHG Protocol — Scope 1 y Scope 2\nColombia (1990–2022)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('MtCO2')
axes[1].legend(fontsize=9)

plt.suptitle('Inventario GHG — Metodología GHG Protocol',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_inventario_ghg.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Regresión y Proyección al 2030 — NDC Colombia

Colombia se comprometió bajo el Acuerdo de París a reducir sus emisiones GHG en un **51% para 2030** respecto al escenario tendencial (BAU — Business As Usual). Este análisis modela la tendencia histórica de emisiones mediante regresión lineal y proyecta el escenario BAU al 2030, comparándolo con la meta NDC para evaluar la brecha de implementación.

In [ ]:
# --- 5.1 Regresión lineal sobre emisiones CO2 Colombia ---
df_reg = df_col[['year', 'co2']].dropna().copy()

X = df_reg[['year']].values
y = df_reg['co2'].values

# Modelo de regresión lineal
modelo_lineal = LinearRegression()
modelo_lineal.fit(X, y)
y_pred = modelo_lineal.predict(X)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
pendiente = modelo_lineal.coef_[0]

print('=== REGRESIÓN LINEAL — EMISIONES CO2 COLOMBIA ===')
print(f'Pendiente:    {pendiente:.3f} MtCO2/año')
print(f'R²:           {r2:.4f}')
print(f'RMSE:         {rmse:.3f} MtCO2')
print()
print(f'Interpretación: Colombia aumenta en promedio {pendiente:.2f} MtCO2 por año')

# Proyección BAU al 2030
años_proyeccion = np.arange(df_reg['year'].max() + 1, 2031).reshape(-1, 1)
co2_proyectado = modelo_lineal.predict(años_proyeccion)

co2_bau_2030 = co2_proyectado[-1]
meta_ndc_2030 = co2_bau_2030 * (1 - 0.51)  # Reducción 51% vs BAU
co2_actual_2022 = df_reg[df_reg['year'] == 2022]['co2'].values[0]

print()
print('=== PROYECCIÓN NDC 2030 ===')
print(f'Escenario BAU 2030:     {co2_bau_2030:.1f} MtCO2')
print(f'Meta NDC 2030 (−51%):   {meta_ndc_2030:.1f} MtCO2')
print(f'Emisiones actuales 2022: {co2_actual_2022:.1f} MtCO2')
print(f'Reducción requerida vs 2022: {co2_actual_2022 - meta_ndc_2030:.1f} MtCO2')

In [ ]:
# --- 5.2 Visualización de proyección ---
fig, ax = plt.subplots(figsize=(14, 7))

# Datos históricos
ax.plot(df_reg['year'], df_reg['co2'],
        color=PALETTE['colombia'], linewidth=2.5,
        label='Emisiones históricas CO2', zorder=5)

# Línea de regresión histórica
ax.plot(df_reg['year'], y_pred,
        color=PALETTE['colombia'], linewidth=1.2,
        linestyle='--', alpha=0.6, label=f'Tendencia lineal (R²={r2:.3f})')

# Proyección BAU
años_hist_proy = np.append(df_reg['year'].values[-1], años_proyeccion.flatten())
co2_hist_proy = np.append(df_reg['co2'].values[-1], co2_proyectado)
ax.plot(años_hist_proy, co2_hist_proy,
        color=PALETTE['alerta'], linewidth=2.2,
        linestyle='-.', label='Escenario BAU (Business As Usual — Tendencia sin acción)')

# Meta NDC
ax.plot([df_reg['year'].max(), 2030],
        [co2_actual_2022, meta_ndc_2030],
        color=PALETTE['verde'], linewidth=2.5,
        label=f'Trayectoria NDC (meta: {meta_ndc_2030:.1f} MtCO2 en 2030)')

# Área de brecha
ax.fill_between([2022, 2030],
                [co2_actual_2022, meta_ndc_2030],
                [co2_actual_2022, co2_bau_2030],
                alpha=0.15, color=PALETTE['alerta'],
                label=f'Brecha de implementación ({co2_bau_2030 - meta_ndc_2030:.1f} MtCO2)')

# Anotaciones
ax.annotate(f'BAU 2030:\n{co2_bau_2030:.1f} MtCO2',
            xy=(2030, co2_bau_2030), xytext=(2027, co2_bau_2030 + 5),
            fontsize=9, color=PALETTE['alerta'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['alerta']))

ax.annotate(f'Meta NDC 2030:\n{meta_ndc_2030:.1f} MtCO2',
            xy=(2030, meta_ndc_2030), xytext=(2027, meta_ndc_2030 - 8),
            fontsize=9, color=PALETTE['verde'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['verde']))

ax.axvline(x=2022, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.text(2022.2, ax.get_ylim()[0] + 2, 'Último dato\ndisponible', fontsize=8, color='gray')

ax.set_title('Proyección de Emisiones CO2 — Colombia 2030\nEscenario BAU vs Meta NDC (−51% vs tendencial)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('MtCO2 (megatoneladas de CO2)', fontsize=11)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(1990, 2031)

plt.tight_layout()
plt.savefig('fig5_proyeccion_ndc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Análisis de Correlación — Emisiones vs Desarrollo Económico

La curva de Kuznets Ambiental (EKC — Environmental Kuznets Curve) postula que a medida que un país se desarrolla económicamente, sus emisiones aumentan inicialmente y luego disminuyen una vez alcanzado un umbral de ingreso. Este análisis evalúa si los países de América Latina siguen ese patrón en emisiones per cápita versus PIB per cápita.

In [ ]:
# --- 6.1 Emisiones vs PIB per cápita — América Latina 2022 ---
df_latam_2022 = df_latam[df_latam['year'] == 2022].copy()
df_latam_2022['gdp_per_capita'] = df_latam_2022['gdp'] / df_latam_2022['population']
df_latam_2022 = df_latam_2022.dropna(subset=['gdp_per_capita', 'co2_per_capita'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter PIB vs emisiones per cápita
for _, row in df_latam_2022.iterrows():
    color = PALETTE['colombia'] if row['country'] == 'Colombia' else PALETTE['neutro']
    size = 120 if row['country'] == 'Colombia' else 60
    axes[0].scatter(row['gdp_per_capita'], row['co2_per_capita'],
                    color=color, s=size, zorder=5, alpha=0.8)
    axes[0].annotate(row['country'],
                     (row['gdp_per_capita'], row['co2_per_capita']),
                     fontsize=7, ha='left', va='bottom',
                     color='black' if row['country'] == 'Colombia' else 'gray')

axes[0].set_title('PIB Per Cápita vs Emisiones CO2 Per Cápita\nAmérica Latina 2022',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('PIB per cápita (USD PPP 2011)')
axes[0].set_ylabel('tCO2 per cápita')

# Participación en emisiones globales — Colombia
df_col_share = df_col[['year', 'share_global_co2']].dropna()
axes[1].fill_between(df_col_share['year'], df_col_share['share_global_co2'],
                     alpha=0.3, color=PALETTE['colombia'])
axes[1].plot(df_col_share['year'], df_col_share['share_global_co2'],
             color=PALETTE['colombia'], linewidth=2.5)
axes[1].set_title('Participación de Colombia en Emisiones\nGlobales de CO2 (%)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('% del total global')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f%%'))

plt.suptitle('Emisiones y Desarrollo Económico — Perspectiva Regional',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_emisiones_pib.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. KPIs de Sostenibilidad — Resumen Ejecutivo

Los siguientes KPIs (Key Performance Indicators — Indicadores Clave de Desempeño) sintetizan el desempeño climático de Colombia en formato de reporte ESG (Environmental, Social and Governance — Ambiental, Social y de Gobernanza), alineado con los estándares GRI (Global Reporting Initiative — Iniciativa Global de Reporte) y el GHG Protocol.

In [ ]:
# --- 7.1 Tabla de KPIs ESG para Colombia ---
año_ref = 1990
año_actual = 2022

def get_val(col, año):
    vals = df_col[df_col['year'] == año][col].values
    return vals[0] if len(vals) > 0 and not np.isnan(vals[0]) else np.nan

kpis = {
    'Emisiones CO2 totales (MtCO2)': {
        '1990': round(get_val('co2', 1990), 2),
        '2022': round(get_val('co2', 2022), 2),
        'Var %': round((get_val('co2', 2022) - get_val('co2', 1990)) / get_val('co2', 1990) * 100, 1)
    },
    'Emisiones per cápita (tCO2/hab)': {
        '1990': round(get_val('co2_per_capita', 1990), 2),
        '2022': round(get_val('co2_per_capita', 2022), 2),
        'Var %': round((get_val('co2_per_capita', 2022) - get_val('co2_per_capita', 1990)) / get_val('co2_per_capita', 1990) * 100, 1)
    },
    'Intensidad carbono (kgCO2/USD PIB)': {
        '1990': round(get_val('co2_per_gdp', 1990), 4),
        '2022': round(get_val('co2_per_gdp', 2022), 4),
        'Var %': round((get_val('co2_per_gdp', 2022) - get_val('co2_per_gdp', 1990)) / get_val('co2_per_gdp', 1990) * 100, 1)
    },
    'GHG totales (MtCO2eq)': {
        '1990': round(get_val('total_ghg', 1990), 2),
        '2022': round(get_val('total_ghg', 2022), 2),
        'Var %': round((get_val('total_ghg', 2022) - get_val('total_ghg', 1990)) / get_val('total_ghg', 1990) * 100, 1)
    },
    'Participación global CO2 (%)': {
        '1990': round(get_val('share_global_co2', 1990), 3),
        '2022': round(get_val('share_global_co2', 2022), 3),
        'Var %': round((get_val('share_global_co2', 2022) - get_val('share_global_co2', 1990)) / get_val('share_global_co2', 1990) * 100, 1)
    }
}

kpi_df = pd.DataFrame(kpis).T.reset_index()
kpi_df.columns = ['Indicador', '1990', '2022', 'Variación (%)']

print('=== RESUMEN EJECUTIVO KPIs ESG — COLOMBIA ===')
print(kpi_df.to_string(index=False))
print()
print(f'Meta NDC 2030: reducir GHG en 51% vs escenario BAU')
print(f'Escenario BAU 2030 proyectado: {co2_bau_2030:.1f} MtCO2')
print(f'Meta NDC 2030: {meta_ndc_2030:.1f} MtCO2')
print(f'Brecha actual: {co2_actual_2022 - meta_ndc_2030:.1f} MtCO2 de reducción requerida')

In [ ]:
# --- 7.2 Exportación para Power BI ---

# Tabla principal Colombia
cols_export = [
    'year', 'co2', 'co2_per_capita', 'co2_per_gdp',
    'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2',
    'methane', 'nitrous_oxide', 'total_ghg',
    'share_global_co2', 'population', 'gdp'
]
df_col[[c for c in cols_export if c in df_col.columns]].to_csv(
    'ghg_colombia_historico.csv', index=False
)

# Tabla comparativa LATAM 2022
cols_latam = ['country', 'year', 'co2', 'co2_per_capita',
              'co2_per_gdp', 'total_ghg', 'share_global_co2']
df_latam[df_latam['year'] == 2022][[c for c in cols_latam if c in df_latam.columns]].to_csv(
    'ghg_latam_2022.csv', index=False
)

# Tabla de proyección NDC
proyeccion_df = pd.DataFrame({
    'year': años_proyeccion.flatten(),
    'co2_bau': co2_proyectado,
    'co2_ndc_meta': np.linspace(co2_actual_2022, meta_ndc_2030,
                                 len(años_proyeccion))
})
proyeccion_df.to_csv('ghg_proyeccion_ndc_2030.csv', index=False)

# Tabla de KPIs
kpi_df.to_csv('ghg_kpis_esg_colombia.csv', index=False)

print('Archivos exportados para Power BI:')
print('  ghg_colombia_historico.csv     — Serie histórica completa Colombia')
print('  ghg_latam_2022.csv             — Comparativa regional 2022')
print('  ghg_proyeccion_ndc_2030.csv    — Escenario BAU vs meta NDC')
print('  ghg_kpis_esg_colombia.csv      — KPIs resumen ejecutivo')

---
## 8. Conclusiones y Recomendaciones

### Hallazgos técnicos

1. Las emisiones de CO2 de Colombia han crecido consistentemente desde 1990, impulsadas principalmente por el consumo de petróleo (oil) y gas natural (gas). La intensidad de carbono (kgCO2/USD PIB) ha disminuido — indicando que la economía colombiana crece más eficientemente en términos de carbono — pero las emisiones absolutas siguen en aumento.

2. Colombia representa aproximadamente el 0.3% de las emisiones globales de CO2, lo que la posiciona como un emisor pequeño a escala mundial pero con alta responsabilidad regional dado su biodiversidad y los compromisos NDC (Nationally Determined Contributions — Contribuciones Determinadas a Nivel Nacional) adquiridos.

3. La regresión lineal sobre la tendencia histórica proyecta un escenario BAU (Business As Usual — Tendencia sin acción climática) para 2030 significativamente por encima de la meta NDC (reducción del 51% vs tendencial). Esto indica que Colombia requiere políticas climáticas adicionales y más agresivas para cumplir sus compromisos del Acuerdo de París.

4. Las emisiones per cápita de Colombia son de las más bajas de América Latina, lo que refleja su matriz energética con alta participación de hidroelectricidad. Sin embargo, los países con mayor PIB per cápita de la región (Argentina, Chile) tienen emisiones per cápita significativamente más altas.

### Recomendaciones de política climática

- Acelerar la transición energética reduciendo dependencia de combustibles fósiles, especialmente petróleo (oil) que es la fuente de mayor crecimiento en emisiones.
- Fortalecer el sistema de MRV (Monitoring, Reporting and Verification — Monitoreo, Reporte y Verificación) de emisiones a nivel sectorial para identificar palancas de reducción específicas.
- Implementar mecanismos de precio al carbono (impuesto o mercado de carbono) para incentivar la reducción de emisiones en el sector industrial.
- Proteger y restaurar ecosistemas forestales como sumideros de carbono naturales, aprovechando el potencial de Colombia como país megadiverso.

### Referencias

- Ritchie, H., Rosado, P. & Roser, M. (2023). *CO₂ and Greenhouse Gas Emissions*. Our World in Data. https://ourworldindata.org/co2-and-greenhouse-gas-emissions
- WRI & WBCSD (2015). *The Greenhouse Gas Protocol: A Corporate Accounting and Reporting Standard*. World Resources Institute.
- UPME (2022). *Factor de Emisión de CO2 de la Red Eléctrica Colombiana*. Unidad de Planeación Minero-Energética.
- DNP (2021). *Contribución Determinada a Nivel Nacional (NDC) de Colombia — Actualización 2020*. Departamento Nacional de Planeación.
- IPCC (2022). *Sixth Assessment Report — Working Group III: Mitigation of Climate Change*. Cambridge University Press.